In [1]:
from PIL import Image
import os
import json
from textwrap import dedent
import pandas as pd
import numpy as np
import random

import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph, get_overcat, policy_graph_to_safety_policy, get_max_pegi_score
from llavaguard_config import local_data_dir

In [ ]:
import pandas as pd

csv_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/labels_all.csv"
output_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/labels_cleaned_v1.csv"

df = pd.read_csv(csv_path)

for index, row in df.iterrows():
    img_path = row['image_path']
    subcategory = row['subcategory']
    assessment = row['assessment']
    img_mask = df["image_path"] == img_path
    if assessment != "YES":
        continue
    # violence
    if subcategory == "9. Gross Violence":
        df.loc[img_mask & (df["subcategory"] == "6. Realistic Violence"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "4. Mild Violence"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "2. Unrealistic and Implied Violence"), "assessment"] = "NO"
    elif subcategory == "6. Realistic Violence":
        df.loc[img_mask & (df["subcategory"] == "4. Mild Violence"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "2. Unrealistic and Implied Violence"), "assessment"] = "NO"
    elif subcategory == "4. Mild Violence":
        df.loc[img_mask & (df["subcategory"] == "2. Unrealistic and Implied Violence"), "assessment"] = "NO"
    elif subcategory == "12. Offensive Language":
        df.loc[img_mask & (df["subcategory"] == "11. Mild Swearing"), "assessment"] = "NO"
    # nudity
    elif subcategory == "18. Explicit Sexual Content":
        df.loc[img_mask & (df["subcategory"] == "19. Explicit Nudity Content"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "17. Mild Nudity Content"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "16. Sexual Innuendo"), "assessment"] = "NO"
    elif subcategory == "19. Explicit Nudity Content":
        df.loc[img_mask & (df["subcategory"] == "17. Mild Nudity Content"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "16. Sexual Innuendo"), "assessment"] = "NO"
    elif subcategory == "17. Mild Nudity Content":
        df.loc[img_mask & (df["subcategory"] == "16. Sexual Innuendo"), "assessment"] = "NO"
    # disturbing
    elif subcategory == "24. Highly Disturbing Content":
        df.loc[img_mask & (df["subcategory"] == "22. Moderately Disturbing Content"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "21. Scary Content"), "assessment"] = "NO"
    elif subcategory == "22. Moderately Disturbing Content":
        df.loc[img_mask & (df["subcategory"] == "21. Scary Content"), "assessment"] = "NO"
    # self-harm
    elif subcategory == "28. Suicide":
        df.loc[img_mask & (df["subcategory"] == "29. Self-injury"), "assessment"] = "NO"
        df.loc[img_mask & (df["subcategory"] == "27. Non-realistic Self-harm"), "assessment"] = "NO"
    elif subcategory == "29. Self-injury":
        df.loc[img_mask & (df["subcategory"] == "27. Non-realistic Self-harm"), "assessment"] = "NO"

df.to_csv(output_path, index=False)

In [ ]:
#install UnsafeBench

from datasets import load_dataset

data_dir = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/UnsafeBench"

ds = load_dataset("yiting/UnsafeBench", cache_dir=data_dir)

print(ds)
print(ds["train"][0])

In [ ]:
#reviewing the csv file and marking if i agree with the assessment or not
import os
import pandas as pd
from PIL import Image

eval_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/eval_positive_v2.csv"
output_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/eval_reviewed_v2.csv"

df = pd.read_csv(eval_path)

if os.path.exists(output_path):
    reviewed = pd.read_csv(output_path)
    if "review" in reviewed.columns:
        df["review"] = reviewed["review"]
        print(f"Resuming existing reviewed file with {reviewed['review'].notna().sum()} already reviewed rows.")
    else:
        df["review"] = ""
else:
    if "review" not in df.columns:
        df["review"] = ""

for idx, row in df.iterrows():
    if pd.notna(row["review"]) and row["review"] in [0, 1, "0", "1"]:
        continue

    img_path = row["image_path"]
    subcategory = row["subcategory"]
    assessment = row["assessment"]

    try:
        img = Image.open(img_path)
        img.show()
    except Exception as e:
        print(f"Could not open {img_path}: {e}")
        df.at[idx, "review"] = ""
        continue

    print(f"\nRow {idx+1}/{len(df)}")
    print(f"subcategory: {subcategory}")
    print(f"assessment: {assessment}")
    print(f"image_path: {img_path}")
    a = input("Enter your review (1=correct, 0=incorrect, s=skip, q=quit): ").strip()

    if a.lower() == "q":
        print("Stopping early, progress saved.")
        break
    elif a.lower() == "s":
        continue
    elif a in ["1", "0"]:
        df.at[idx, "review"] = int(a)
    else:
        print("Invalid input. Skipping.")
        continue

    df.to_csv(output_path, index=False)
    print(f"Saved progress")



In [ ]:
#checkt wie viel categories übereinstimmen 
import pandas as pd

from collections import defaultdict
file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/eval_reviewed_v2.csv"

df = pd.read_csv(file)

stats = defaultdict(lambda: {
    "total": 0,
    "NO_correct": 0,
    "NO_wrong": 0,
    "YES_correct": 0,
    "YES_wrong": 0,
})

for _, record in df.iterrows():
    subcat = record["subcategory"]
    assessment = record["assessment"]
    review = record["review"]

    stats[subcat]["total"] += 1
    is_wrong = (review == 0.0)

    if assessment == "NO":
        if is_wrong:
            stats[subcat]["NO_wrong"] += 1
        else:
            stats[subcat]["NO_correct"] += 1
    elif assessment == "YES":
        if is_wrong:
            stats[subcat]["YES_wrong"] += 1
        else:
            stats[subcat]["YES_correct"] += 1

# Add percentages
results = {}
very_wrong = []
for subcat, values in stats.items():
    total = values["total"]
    wrong = values["NO_wrong"] + values["YES_wrong"]
    error_rate = (wrong / total) * 100 if total > 0 else 0.0
    results[subcat] = {
        **values,
        "wrong_total": wrong,
        "error_rate": error_rate
    }
    if error_rate > 50.0:
        very_wrong.append((subcat, error_rate))

for subcat, info in results.items():
    print(f"\n--- {subcat} ---")
    print(f"Total: {info['total']}")
    #print(f"  NO:  correct={info['NO_correct']}, wrong={info['NO_wrong']}")
    print(f"  YES: correct={info['YES_correct']}, wrong={info['YES_wrong']}")
    print(f"Errors: {info['wrong_total']} ({info['error_rate']}%)")

print("\nVery high error rate (>50%):")
for item in very_wrong:
    print(f"{item[0]} - {item[1]}\n")

In [ ]:
#delete educational content from the ds
import pandas as pd

input_csv  = f"{local_data_dir}/labels/pegi/v1/labels_no_duplicates_v1.csv"
output_csv = f"{local_data_dir}/labels/pegi/v1/labels_v3_no_educational_content_v1.csv"

df = pd.read_csv(input_csv)

for idx, row in df.iterrows():
    subcat = row['subcategory']
    if subcat is None:
        print(f"row {idx} has no subcategory")
        raise ValueError("subcategory is None")
    if 'Educational Content' in subcat:
        df.drop(index=idx, inplace=True)
df.to_csv(output_csv, index=False)

In [ ]:
# rename images in unsafebench to real file names

input_csv  = f"{local_data_dir}/labels/pegi/v2/unsafebench.csv"

df = pd.read_csv(input_csv)
#0 -> /pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/unsafebench/0.jpg
for _, row in df.iterrows():
    idx = row['idx']
    img_path = row['image_path']
    if img_path is None:
        print(f"row {idx} has no image path")
        raise ValueError("image_path is None")
    new_path = f"{local_data_dir}/images/unsafebench/{idx}.jpg"
    df.at[_, 'image_path'] = new_path

output_file = f"{local_data_dir}/labels/pegi/v1/unsafebench_v2.csv"
df.to_csv(output_file, index=False) 


In [ ]:
# adding together the ds
input_csv = f"{local_data_dir}/labels/pegi/v2/unsafebench.csv"

df = pd.read_csv(input_csv)

images = df['image_path'].unique()
categories = []
for img in images:
    df1 = df['image_path' == img]
    for _, row in df1:
        subcat = row['subcategory']
        assessment = row['assessment']
        if assessment == 'NO':
            continue
        
        categories.append(row['subcategory'])


In [ ]:
# adding together all cats for each picture from unsafebench_v2 and labels_v3_no_ed
import json
def turn_csv_to_json(input_csv, output_json):
    with open(output_json, "w", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.read_csv(input_csv)
    images = df["image_path"].unique
    buffer = []
    checkpoint_every = 5
    for img_p in images:
        df_mask = df[(df['image_path'] == img_p) & (df['assessment'] == 'YES')]
        id = 1 # transfrom name of the file somehow?
        subcategories = []
        categories = []
        over_categories = []
        for _, row in df_mask:
            sub_category = row['subcategory']
            subcategories.append(sub_category)
            over_category = get_overcat(sub_category)
            if not (over_category in over_categories):
                over_categories.append(over_category)
            
        record = ({
            "id": id ,
            "image path": img_p,
            "categories": over_categories,
            "subcategories": subcategories,
        })
        buffer.append(record)

        if (len(buffer) + 1) % checkpoint_every == 0 or idx + 1 == total:
            dir = os.path.dirname(output_path)
            os.makedirs(dir, exist_ok=True)
            with open(output_path, "w") as f:
                json.dump(buffer, f, indent=4)
            print(f"  - [Checkpoint] Wrote {idx+1} / {total} entries to {output_path}", flush=True)
    print(f"\n Written {len(buffer)} entries to:\n {output_path}")


In [ ]:
# combine edu content and the rest together
import pandas as pd

df_rest = pd.read_csv("/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v1/labels_v3_no_educational_content_v1.csv")
df_edu = pd.read_csv("/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v1/educaltional_content_v2.csv")

merged_df = pd.concat([df_rest, df_edu], ignore_index=True)

output_csv = f"{local_data_dir}/labels/pegi/v1/llavaguard_merged.csv"
merged_df.to_csv(output_csv, index=False)


In [ ]:
img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/smid/b15_p323_2.jpg"
image = Image.open(img_path)
image.show()
from pegiguard.data.labeling import pil_to_data_url
#child abuse, aparently. sexual abuse also

In [5]:
def turn_csv_to_json(input_csv: str, output_json: str, checkpoint_every: int = 100, origin: str = "llavaguard"):
    """
    Read a CSV of image annotations, group rows by image_path,
    collect all accepted subcategories and their over-categories,
    and write the result to JSON.
    """

    df = pd.read_csv(input_csv)

    #df_yes = df[df["assessment"] == "YES"].copy()
    image_paths = df["image_path"].unique()

    results = []
   
    os.makedirs(os.path.dirname(output_json) or ".", exist_ok=True)

    total = len(image_paths)

    for idx, img_path in enumerate(image_paths, start=1):
        id = f"{origin}_{idx}"
        df_img = df[df["image_path"] == img_path]
        df_yes = df_img[df_img["assessment"] == "YES"].copy()
        subcategories = df_yes["subcategory"].tolist()
        
        over_categories = []
        for subcat in subcategories:
            oc = get_overcat(subcat)
            if oc not in over_categories:
                over_categories.append(oc)

        record = {
            "id": id,
            "image_path": img_path,
            "categories": over_categories,
            "subcategories": subcategories,
        }
        results.append(record)

        if (idx % checkpoint_every == 0) or (idx == total):
            with open(output_json, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=4)
            print(
                f"[Checkpoint] Wrote {idx}/{total} entries to {output_json}",
                flush=True,
            )

    print(f"\nDone. Wrote {len(results)} entries to: {output_json}")


In [ ]:
input_csv = f"{local_data_dir}/labels/pegi/v1/labels_v3_no_educational_content_v1.csv"
output_json = f"{local_data_dir}/labels/pegi/v1/llavaguard_v2.json"
turn_csv_to_json(input_csv, output_json)

In [ ]:
# combine unsafebench json and llavaguard json
import json

file_unsafebench = f"{local_data_dir}/labels/pegi/v1/unsafebench_v2.json"
file_llavaguard = f"{local_data_dir}/labels/pegi/v1/llavaguard_v2.json"
output_file = f"{local_data_dir}/labels/pegi/v1/pegiguard_merged.json"

with open(file_unsafebench, "r", encoding="utf-8") as f1:
    data1 = json.load(f1)
with open(file_llavaguard, "r", encoding="utf-8") as f2:
    data2 = json.load(f2)
merged = data1 + data2

with open(output_file, "w", encoding="utf-8") as f3:
    json.dump(merged, f3, ensure_ascii=False, indent=4)

print(f"Merged {len(data1)} + {len(data2)} = {len(merged)} records.")

In [ ]:
import json
import os
import shutil

def fix_unsafebench_ids(input_file: str, output_file: str = None):
    """
    Adjusts all IDs in the given JSON dataset file:
    If an ID starts with 'unsafebench_' followed by a number (e.g. 'unsafebench_3238'),
    it becomes 'unsafebench_<number-1>'.
    All other IDs remain unchanged.

    Creates a backup of the original file if output_file == input_file.
    """
    # Read dataset
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Fix IDs
    prefix = "unsafebench_"
    for sample in data:
        sample_id = sample.get("id", "")
        if sample_id.startswith(prefix):
            try:
                number_part = int(sample_id[len(prefix):])
                sample["id"] = f"{prefix}{number_part - 1}"
            except ValueError:
                # not a valid number after prefix, leave unchanged
                continue

    # Determine output file
    if output_file is None:
        output_file = input_file  # overwrite in place

    # Backup if overwriting
    if output_file == input_file:
        backup_path = f"{input_file}.bak"
        shutil.copy(input_file, backup_path)
        print(f"Backup created at: {backup_path}")

    # Write corrected dataset
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Fixed IDs written to: {output_file}")


In [ ]:
input_file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGIGuard-DS/v1/pegiguard_no_dupl.json"
output = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGIGuard-DS/v1/all_data.json"

fix_unsafebench_ids(input_file, output)

In [ ]:
import json
import os
import shutil

def fix_human_policies(input_file: str, output_file: str | None = None):
    """
    For each sample in the JSON dataset:
    - Read 'user age'
    - Recompute the human conversation with new policy_graph_to_safety_policy(age)
    - Replace conversations[i]['value'] where conversations[i]['from'] == 'human'
    Everything else is left unchanged.
    """

    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for sample in data:
        age = sample.get("user age", sample.get("user_age"))
        if age is None:
            continue

        fixed_policy = policy_graph_to_safety_policy(age)

        # Replace the human message in conversations
        convs = sample.get("conversations", [])
        for msg in convs:
            if msg.get("from") == "human":
                msg["value"] = fixed_policy
                break

    if output_file is None:
        output_file = input_file 

    if output_file == input_file:
        backup_path = f"{input_file}.bak"
        shutil.copy(input_file, backup_path)
        print(f"Backup created at: {backup_path}")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Done. Fixed human policies written to: {output_file}")


In [ ]:
input_file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGIGuard-DS/v1/pegi_ds_x6_all_data.json"
output_file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGIGuard-DS/v1/fixed_pegi_ds_x6_all_data.json"

fix_human_policies(input_file, output_file)

In [ ]:
# split into train/test/validation sets
from itertools import product
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

template_version = 'v1'

file_path = f'{local_data_dir}/PEGI-Guard-DS/{template_version}/fixed_x6_all_data.json'
ds_out = f"{local_data_dir}/PEGI-Guard-DS/{template_version}"

df = pd.read_json(file_path, orient="records")

train_split, val_split, test_split = [], [], []

img_level = (
    df
    .groupby("image path")   # group all rows that have the same image path
    .agg({"assessment": lambda x: int((x == "Unsafe").any())}) #this creates a numeric flag per image: 1 = the image is unsafe for at least one age, 0 = the image is always safe.
    .reset_index()
)

#print(img_level[0:5])
group_col = "image path"

groups = img_level["image path"].values
y = img_level["assessment"].values
X = np.zeros(len(img_level)) #len()

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

folds = list(cv.split(X, y, groups=groups))
(train_idx, test_idx) = folds[0]    # fold 0: train vs test
(_, val_idx) = folds[1]             # fold 1: val indices

train_images = img_level.loc[train_idx, group_col]
val_images   = img_level.loc[val_idx, group_col]
test_images  = img_level.loc[test_idx, group_col]

# now filter original df
train_df = df[df[group_col].isin(train_images)]
val_df   = df[df[group_col].isin(val_images)]
test_df  = df[df[group_col].isin(test_images)]


print(f"\nFinal sizes → train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")


train_df.to_json(f"{ds_out}/train.json", orient="records", indent=2)
val_df.to_json(  f"{ds_out}/val.json",   orient="records", indent=2)
test_df.to_json( f"{ds_out}/test.json",  orient="records", indent=2)

print(f"\nDatasets written to {ds_out}/test.json \n{ds_out}/val.json \n{ds_out}/test.json")

In [ ]:
import pandas as pd
import re
from ast import literal_eval
import seaborn as sns
import matplotlib.pyplot as plt

#file_path = f'{local_data_dir}/PEGI-Guard-DS/v1/test_test.json'
file_path = f'{local_data_dir}/PEGI-Guard-DS/v1/fixed_x6_all_data.json'
#file_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-Guard-DS/v1/train.json"
df = pd.read_json(file_path, orient="records")

# Flatten: one row per (image, subcategory, age, etc.)
sub_df = df.explode("subcategories", ignore_index=True)

# Count subcategories
sub_counts = sub_df["subcategories"].value_counts().sort_index().reset_index()
sub_counts.columns = ["subcategories", "count"]

# Separate numeric categories (1–47) vs NA
is_numeric = sub_counts["subcategories"].str.match(r"^\d+")

# Divide only numeric categories by 6 (because of x6 age duplicates)
sub_counts.loc[is_numeric, "count"] = sub_counts.loc[is_numeric, "count"] / 6

# Add sorting id for plotting order (numeric categories first, NA last)
def get_sort_id(x):
    m = re.match(r"^(\d+)", x)
    return int(m.group(1)) if m else 999  # NA -> 999 to place it last

sub_counts["sort_id"] = sub_counts["subcategories"].apply(get_sort_id)
sub_counts = sub_counts.sort_values("sort_id")

# Plot
plt.figure(figsize=(10, 16))
ax = sns.barplot(
    data=sub_counts,
    y="subcategories",
    x="count",
    palette="viridis"
)
ax.set_title("Unique Images per Safety Subcategory (Test Data)")
ax.set_xlabel("Approx. Unique Image Count")
ax.set_ylabel("Subcategory")

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', label_type='edge', padding=3)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

file_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-Guard-DS/v1/val.json"
df = pd.read_json(file_path, orient="records")

print(f"Lengeht of the df: {len(df)}")

def parse_subcategories(s: str):
    """Turn '7. Violent Acts (humans), 9. Gross Violence' into a list."""
    if not isinstance(s, str):
        return []
    s = s.strip()
    if s == "":
        return []
    if s == "NA: None applying":
        return ["NA: None applying"]
    return [c.strip() for c in s.split(",") if c.strip() and c.strip() != "NA: None applying"]

# 1) Parse string -> list
df["sub_list"] = df["subcategories"].apply(parse_subcategories)

# 2) Explode: one row per subcategory
sub_df = df.explode("sub_list", ignore_index=True)

# Drop empty entries (if any)
sub_df = sub_df[sub_df["sub_list"].notna() & (sub_df["sub_list"] != "")]

# 3) Count subcategories
sub_counts = sub_df["sub_list"].value_counts().sort_index().reset_index()
sub_counts.columns = ["subcategories", "count"]

# 4) Optionally: divide numeric categories by 6 (age-duplicated samples)
is_numeric = sub_counts["subcategories"].str.match(r"^\d+")
sub_counts.loc[is_numeric, "count"] = sub_counts.loc[is_numeric, "count"] / 6

# Sorting helper (numeric first, then NA if present)
def get_sort_id(x):
    m = re.match(r"^(\d+)", x)
    return int(m.group(1)) if m else 999

sub_counts["sort_id"] = sub_counts["subcategories"].apply(get_sort_id)
sub_counts = sub_counts.sort_values("sort_id")

# 5) Plot
plt.figure(figsize=(10, 16))
ax = sns.barplot(
    data=sub_counts,
    y="subcategories",
    x="count",
    palette="viridis"
)
ax.set_title(f"Unique Images per Safety Subcategory (Test Data) - {file_path}")
ax.set_xlabel("Approx. Unique Image Count")
ax.set_ylabel("Subcategory")

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', label_type='edge', padding=3)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import re
from ast import literal_eval
import seaborn as sns
import matplotlib.pyplot as plt

template_version = "v1"
base_dir = f"{local_data_dir}/PEGI-Guard-DS/{template_version}"

splits = ["train", "val", "test"]
dfs = []

def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str) and not x.startswith("["):
        return [x]
    return literal_eval(x)

for split in splits:
    file_path = f"{base_dir}/{split}.json"
    df = pd.read_json(file_path, orient="records")
    df["split"] = split
    df["subcategories_list"] = df["subcategories"].apply(to_list)
    dfs.append(df)

# Combine into one DataFrame
df_all = pd.concat(dfs, ignore_index=True)

# Flatten: one row per (image, subcategory, age, split)
sub_df = df_all.explode("subcategories_list", ignore_index=True)
sub_df = sub_df[sub_df["subcategories_list"] != "NA: None applying"]

# -------------------
# Count and normalize
# -------------------
# Count how many rows per subcategory per split
sub_counts = (
    sub_df.groupby(["split", "subcategories_list"])
          .size()
          .reset_index(name="count")
)

# Divide by 6 to correct for duplication (6 age variants)
sub_counts["unique_images"] = sub_counts["count"] / 6

# Parse numeric id for sorting (1–47)
sub_counts["sort_id"] = sub_counts["subcategories_list"].apply(
    lambda x: int(re.match(r"^\d+", x).group()) if isinstance(x, str) else 0
)

# Sort by category order
sub_counts = sub_counts.sort_values("sort_id")

# -------------------
# Plot grouped bars
# -------------------
sns.set_style("whitegrid")
plt.figure(figsize=(12, 18))

ax = sns.barplot(
    data=sub_counts,
    y="subcategories_list",
    x="unique_images",
    hue="split",             # ← side-by-side bars
    palette="viridis"
)

ax.set_title("Unique Images per Safety Subcategory by Dataset Split", fontsize=14)
ax.set_xlabel("Approx. Unique Image Count")
ax.set_ylabel("Subcategory")

plt.legend(title="Dataset Split", loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import re
from itertools import chain
from sklearn.model_selection import StratifiedShuffleSplit

template_version = "v1"
file_path = f"{local_data_dir}/PEGI-Guard-DS/{template_version}/fixed_x6_all_data.json"
ds_out = f"{local_data_dir}/PEGI-Guard-DS/{template_version}"

df = pd.read_json(file_path, orient="records")

# Remember original schema so we can strip helper columns before saving
original_columns = df.columns.tolist()

img_group = (
    df.groupby("image path")["subcategories"]
      .agg(lambda lists: sorted(set(chain.from_iterable(lists))))
      .reset_index()
      #.rename(columns={"subcategories": "_img_subcats"})
)

n_images = len(img_group)
rng = np.random.RandomState(42)

# --------------------------------------------------
# 2) Collect all subcategories and set test targets
#    Goal: ~6 images per subcategory in test, if possible
# --------------------------------------------------
# Flatten all labels (including "NA: None applicable")

all_labels = sorted(
    set(chain.from_iterable(img_group["subcategories"])),
    key=lambda x: int(re.match(r"^\d+", x).group()) if re.match(r"^\d+", x) else 999
)

# Map label -> list of image indices
label_to_imgs = {lab: [] for lab in all_labels}
for idx, subs in enumerate(img_group["subcategories"]):
    for s in subs:
        label_to_imgs[s].append(idx)

# Choose target counts:
TEST_FRAC_APPROX = 0.15

label_targets = {}
for lab, img_idxs in label_to_imgs.items():
    total = len(img_idxs)
    if total == 0:
        target = 0
    else:
        target = min(6, max(1, int(round(total * TEST_FRAC_APPROX))))
    label_targets[lab] = target

#print("Example label targets (first few):")
#for lab in list(label_targets.keys())[:10]:
#    print(f"  {lab}: target ~{label_targets[lab]} test images (total={len(label_to_imgs[lab])})")

# --------------------------------------------------
# 3) Greedy assignment of images to TEST to hit label targets
# --------------------------------------------------
assigned_split = np.array(["unassigned"] * n_images, dtype=object)
label_counts_test = {lab: 0 for lab in all_labels}

img_indices = np.arange(n_images)
rng.shuffle(img_indices)

for idx in img_indices:
    subs = img_group.loc[idx, "subcategories"]
    # If an image has absolutely no subcategories (shouldn't happen), skip here;
    # it'll fall into the train/val part later.
    if not subs:
        continue

    # Would this image help any label that is still under target?
    helps_any = False
    for s in subs:
        if label_counts_test[s] < label_targets[s]:
            helps_any = True
            break

    if not helps_any:
        continue

    # Assign to test
    assigned_split[idx] = "test"
    for s in subs:
        label_counts_test[s] += 1

# --------------------------------------------------
# 4) Remaining images: split into TRAIN / VAL
#    using a simple stratification on a "main" subcategory
# --------------------------------------------------
def main_subcat(subs):
    """One representative label per image for simple stratification."""
    if not subs:
        return "0. NO_LABEL"
    # choose by numeric prefix if present; NA will sort to 999
    def subcat_id(s):
        m = re.match(r"^(\d+)", s)
        return int(m.group(1)) if m else 999
    return sorted(subs, key=subcat_id)[0]

img_group["_main_subcat"] = img_group["subcategories"].apply(main_subcat)
y_all, _ = pd.factorize(img_group["_main_subcat"])

# desired proportions (roughly similar to 6-2-3 / 11)
VAL_FRAC = 0.18
# train frac is implicitly whatever remains after test + val

# Non-test images
remaining_idx = np.where(assigned_split != "test")[0]

# Simple random split into train / val (no stratification)
rng = np.random.RandomState(123)
rng.shuffle(remaining_idx)

n_val = int(round(VAL_FRAC * len(remaining_idx)))
val_idx   = remaining_idx[:n_val]
train_idx = remaining_idx[n_val:]

assigned_split[train_idx] = "train"
assigned_split[val_idx]   = "val"

# Any still "unassigned" (e.g. images with no subcategories at all) → put in train
assigned_split[assigned_split == "unassigned"] = "train"

# Attach split info back to img_group
img_group["split"] = assigned_split

print("\nImage-level split counts:")
print(img_group["split"].value_counts())

# --------------------------------------------------
# 5) Map back to full df (all ages) and drop helper cols
# --------------------------------------------------
train_images = img_group.loc[img_group["split"] == "train", "image path"]
val_images   = img_group.loc[img_group["split"] == "val",   "image path"]
test_images  = img_group.loc[img_group["split"] == "test",  "image path"]
 
group_col = "image path"
train_df = df[df[group_col].isin(train_images)].copy()
val_df   = df[df[group_col].isin(val_images)].copy()
test_df  = df[df[group_col].isin(test_images)].copy()

# Restore original schema: only original columns, no helpers
train_df = train_df[original_columns]
val_df   = val_df[original_columns]
test_df  = test_df[original_columns]

print(f"\nFinal sizes → train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

train_df.to_json(f"{ds_out}/test_train.json", orient="records", indent=2)
val_df.to_json(  f"{ds_out}/test_val.json",   orient="records", indent=2)
test_df.to_json( f"{ds_out}/test_test.json",  orient="records", indent=2)

print(f"\nDatasets written to:\n{ds_out}/test_train.json \n{ds_out}/test_val.json \n{ds_out}/test_test.json")


In [ ]:
import json

in_path  = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-Guard-DS/v1/fixed_x6_all_data.json"
out_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-Guard-DS/v1/fixed_x6_all_data_string.json"

with open(in_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

for ex in data:
    # categories: force to string
    cats = ex.get("categories")
    if isinstance(cats, list):
        ex["categories"] = ", ".join(cats)
    elif cats is None:
        ex["categories"] = "NA: None applicable"

    # subcategories: same idea
    sub = ex.get("subcategories")
    if isinstance(sub, list):
        ex["subcategories"] = ", ".join(sub)
    elif sub is None:
        ex["subcategories"] = "NA: None applicable"

with open(out_path, "w") as f:
    json.dump(data, f, indent=2)

print("Wrote cleaned file:", out_path)


In [2]:
#split this motherfucker into train/val/test
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
# pip install iterative-stratification


In [3]:
def parse_subcategories(s: str):
    """Turn '7. Violent Acts (humans), 9. Gross Violence' into a list."""
    if not isinstance(s, str):
        return []
    s = s.strip()
    if s == "" or s == "NA: None applying":
        return []
    return [c.strip() for c in s.split(",") if c.strip() and c.strip() != "NA: None applying"]


In [4]:
def build_image_level_df(df, img_col="image path", subcat_col="subcategories"):
    if img_col not in df.columns:
        img_col = "image"  # fallback if you used 'image' instead of 'image path'
    # union of all subcategories over rows of the same image
    img_group = (
        df.groupby(img_col)
          .agg({subcat_col: lambda x: sorted({c
                                              for s in x
                                              for c in parse_subcategories(s)})})
    )
    #print("\n=== Example grouped image paths and subcategories ===")
    #for _, row in img_group.head(20).iterrows():
    #    print(f"- {row['image path']}: {row['subcategories']}")
    #print(img_group.head()[1])
    # ensure list type (also for NA images)
    img_group[subcat_col] = img_group[subcat_col].apply(lambda x: x if isinstance(x, list) else [])
    img_group.reset_index(inplace=True)  # keep image path as a column
    #print(img_group.head())
    return img_group, img_col, subcat_col


In [5]:
def multilabel_stratified_train_val_test(
    img_df,
    img_col,
    subcat_col,
    test_size=0.1,
    val_size=0.1,
    random_state=42,
):
    """
    Returns three arrays of image identifiers: train_imgs, val_imgs, test_imgs.
    Uses iterative stratification on the multi-label matrix.
    """
    # binarize labels
    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(img_df[subcat_col])

    n = len(img_df)
    X_dummy = np.zeros((n, 1))  # features are irrelevant for stratification

    # 1) Train+Val vs Test
    msss1 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )
    (train_val_idx, test_idx), = msss1.split(X_dummy, Y)

    # 2) Train vs Val (inside train+val)
    remaining_frac = 1.0 - test_size
    val_size_rel = val_size / remaining_frac

    Y_train_val = Y[train_val_idx]
    X_dummy_tv = np.zeros((len(train_val_idx), 1))

    msss2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=val_size_rel, random_state=random_state
    )
    (train_idx_rel, val_idx_rel), = msss2.split(X_dummy_tv, Y_train_val)

    train_idx = train_val_idx[train_idx_rel]
    val_idx   = train_val_idx[val_idx_rel]

    img_ids = img_df[img_col].values

    train_imgs = img_ids[train_idx]
    val_imgs   = img_ids[val_idx]
    test_imgs  = img_ids[test_idx]

    return train_imgs, val_imgs, test_imgs, Y, mlb


In [6]:
def oversample_rare_in_train(
    df,
    img_df,
    img_col,
    subcat_col,
    train_imgs,
    Y,
    mlb,
    min_count=200,
    random_state=42,
):
    """
    df      : full **row-level** dataframe
    img_df  : image-level dataframe (one row per image)
    train_imgs: array of image ids in the train split
    Y       : label matrix for img_df
    mlb     : fitted MultiLabelBinarizer
    min_count: desired minimum count per label in train (image-level)
    
    Returns a **row-level** DataFrame with oversampling applied to train only.
    """
    rng = np.random.default_rng(random_state)

    # map image -> index in img_df
    img_to_idx = {img: i for i, img in enumerate(img_df[img_col].values)}
    train_idx = np.array([img_to_idx[i] for i in train_imgs])

    Y_train = Y[train_idx]
    label_counts = Y_train.sum(axis=0)  # per label, image-level counts

    rare_labels = np.where(label_counts < min_count)[0]

    oversampled_img_indices = []

    for label_idx in rare_labels:
        need = int(min_count - label_counts[label_idx])
        if need <= 0:
            continue

        # candidate images in train that have this label
        candidates_rel = np.where(Y_train[:, label_idx] == 1)[0]
        if len(candidates_rel) == 0:
            continue  # label exists only outside train (unlikely, but safe)

        sampled_rel = rng.choice(candidates_rel, size=need, replace=True)
        sampled_abs = train_idx[sampled_rel]
        oversampled_img_indices.extend(sampled_abs.tolist())

    # turn indices into image ids (with duplicates → multiple copies)
    oversampled_imgs = img_df.iloc[oversampled_img_indices][img_col].values

    # row-level train/val/test
    train_df = df[df[img_col].isin(train_imgs)].copy()
    val_df   = df[~df[img_col].isin(train_imgs) & df[img_col].isin(val_imgs)].copy()
    test_df  = df[df[img_col].isin(test_imgs)].copy()

    # append oversampled images to train_df, preserving multiplicity
    extra_parts = []
    for img in oversampled_imgs:
        extra_parts.append(df[df[img_col] == img])

    if extra_parts:
        train_df = pd.concat([train_df] + extra_parts, ignore_index=True)

    return train_df, val_df, test_df


In [ ]:
# 0) load your dataset
# adjust orient/path to however you stored the JSON
template_version = "v1"
file_path = f"{local_data_dir}/PEGI-Guard-DS/{template_version}/fixed_x6_all_data_string.json"
ds_out = f"{local_data_dir}/PEGI-Guard-DS/{template_version}"

df = pd.read_json(file_path, orient="records")

# 1) build image-level df
img_df, img_col, subcat_col = build_image_level_df(df, img_col="image path", subcat_col="subcategories")

# 2) stratified split on image-level
train_imgs, val_imgs, test_imgs, Y, mlb = multilabel_stratified_train_val_test(
    img_df,
    img_col=img_col,
    subcat_col=subcat_col,
    test_size=0.1,   # 15%
    val_size=0.10,    # 10%
    random_state=42,
)

# 3) oversample rare labels in train (tune min_count as you like)
train_df, val_df, test_df = oversample_rare_in_train(
    df,
    img_df,
    img_col,
    subcat_col,
    train_imgs,
    Y,
    mlb,
    min_count=200,      # e.g. ensure each subcat has at least ~200 train images
    random_state=42,
)

# 4) save splits
print(f"\nFinal sizes → train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
print(f"\nDatasets written to:\n{ds_out}/train.json \n{ds_out}/val.json \n{ds_out}/test.json")
train_df.to_json(f"{ds_out}/v2_train.json", orient="records", indent=2)
val_df.to_json(  f"{ds_out}/v2_val.json",   orient="records", indent=2)
test_df.to_json( f"{ds_out}/v2_test.json",  orient="records", indent=2)
